In [ ]:
import pandas as pd
from modelens import RegressionAnalyzer

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_excel('dataset.xls')
df = raw_data.copy()

#### 1 - EDA + ETL

In [ ]:
df = df.rename(columns={
    "Cement (component 1)(kg in a m^3 mixture)":"Cement",
    "Blast Furnace Slag (component 2)(kg in a m^3 mixture)":"Blast Furnace Slag",
    "Fly Ash (component 3)(kg in a m^3 mixture)":"Fly Ash",
    "Water  (component 4)(kg in a m^3 mixture)":"Water",
    "Superplasticizer (component 5)(kg in a m^3 mixture)":"Super Plasticizer",
    "Coarse Aggregate  (component 6)(kg in a m^3 mixture)":"Coarse Aggregate",
    "Fine Aggregate (component 7)(kg in a m^3 mixture)":"Fine Aggregate",
    "Age (day)":"Age",
    "Concrete compressive strength(MPa, megapascals) ":"Concrete Compressive Strength",
})

In [ ]:
analyzer = RegressionAnalyzer(df,target='Concrete Compressive Strength')

In [ ]:
analyzer.info()

In [ ]:
df = df.drop_duplicates()

In [ ]:
analyzer.reinit(df=df,target='Concrete Compressive Strength')

In [ ]:
analyzer.info()

In [ ]:
df.head()

#### 2 - Comparing Models

In [ ]:
target = 'Concrete Compressive Strength'
X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop(target)
features

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso()),
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet()),
    ]),

    # Distance / Kernel
    "KNN": Pipeline([
        ("scaler",StandardScaler()),
        ("KNN",KNeighborsRegressor())
    ]),
    "SVR":Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR()),
    ]),

    # Tree
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
    ),

    # Bagging
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    # Boosting
    "AdaBoost": AdaBoostRegressor(
        random_state=42,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        random_state=42,
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0,
    ),
}
compare = analyzer.compare_models(models=models,features=features,export_html=True)

#### 3 - Evaluate Data

In [ ]:
_,suspicious_features=analyzer.correlation(features=features)

In [ ]:
analyzer.vif(features=features)

In [ ]:
suspicious_features = ['Cement','Blast Furnace Slag','Fine Aggregate','Water','Fly Ash']

analyzer.evaluate_single_feature_removal(features=suspicious_features,export_html=True)

In [ ]:
selected_model = LGBMRegressor()
analyzer.evaluate_single_feature_removal(model=selected_model,features=features,export_html=True)

In [ ]:
candidates = suspicious_features
analyzer.evaluate_feature_removal_combinations(model=selected_model,features=features,candidates=candidates,export_html=True)

In [ ]:
analyzer.permutation_importance(model=selected_model,export_html=True)

In [ ]:
analyzer.check_overfitting(model=selected_model)

In [ ]:
analyzer.residual_analysis(model=selected_model,export_html=True)

In [ ]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 5, 10],
    "min_child_samples": [10, 20, 40],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
analyzer.tune_model(model=selected_model,param_grid=param_grid,export_html=True)

In [ ]:
analyzer.learning_curve(model=selected_model)

In [26]:
#{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_samples': 10, 'n_estimators': 500, 'num_leaves': 15, 'subsample': 1.0}

models = {
    "LightGBM": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),
    "LightGBM Tune": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        colsample_bytree= 0.8, learning_rate= 0.1, max_depth= 10, min_child_samples= 10, n_estimators= 500, num_leaves= 15, subsample= 1.0
    ),
    
}
compare = analyzer.compare_models(models=models,features=features,export_html=True,html_path='html_reports/compare_final_models.html')